In [92]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from itertools import product
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

DATA_ROOT = Path('/home/igor/noise_scaling/data')

ALGOS = ['Geneformer', 'PCA', 'RandomProjection', 'SCVI']

EXPECTED = {
    'PBMC': {
        'sizes': [100, 215, 464, 1000, 2154, 4641, 10000, 21544, 46415, 100000],
        'qualities': [0.0012346, 0.0025982, 0.0054682, 0.0115083, 0.02422, 0.050973, 0.1072766, 0.225772, 0.4751547, 1.0],
        'signals': ['protein_counts'],
        'seeds': [42, 2303, 2701],
    },
    'larry': {
        'sizes': [100, 215, 464, 1000, 2154, 4641, 10000, 21544, 46415, 100000],
        'qualities': [0.003876, 0.0071835, 0.0133136, 0.0246748, 0.0457311, 0.0847557, 0.1570821, 0.2911284, 0.5395631, 1.0],
        'signals': ['clone'],
        'seeds': [42, 1404, 2701],
    },
    'merfish': {
        'sizes': [100, 203, 414, 843, 1716, 3494, 7113, 14480, 29475, 60000],
        'qualities': [0.027248, 0.0406617, 0.0606789, 0.0905502, 0.1351267, 0.2016475, 0.3009156, 0.4490518, 0.6701133, 1.0],
        'signals': ['ng_idx'],
        'seeds': [1404, 2303, 2701],
    },
    'shendure': {
        'sizes': [100, 359, 1291, 4641, 16681, 59948, 215443, 774263, 2782559, 10000000],
        'qualities': [0.004, 0.0073875, 0.0136438, 0.0251984, 0.0465384, 0.0859506, 0.1587401, 0.2931733, 0.5414548, 1.0],
        'signals': ['author_day'],
        'seeds': [42],
    },
}

for ds, cfg in EXPECTED.items():
    n_expected = len(cfg['sizes']) * len(cfg['qualities']) * len(ALGOS) * len(cfg['signals']) * len(cfg['seeds'])
    print(f"{ds}: {len(cfg['sizes'])} sizes x {len(cfg['qualities'])} qualities x {len(ALGOS)} algos "
          f"x {len(cfg['signals'])} signal ({cfg['signals']}) x {len(cfg['seeds'])} seeds ({cfg['seeds']}) "
          f"= {n_expected} expected")

PBMC: 10 sizes x 10 qualities x 4 algos x 1 signal (['protein_counts']) x 3 seeds ([42, 2303, 2701]) = 1200 expected
larry: 10 sizes x 10 qualities x 4 algos x 1 signal (['clone']) x 3 seeds ([42, 1404, 2701]) = 1200 expected
merfish: 10 sizes x 10 qualities x 4 algos x 1 signal (['ng_idx']) x 3 seeds ([1404, 2303, 2701]) = 1200 expected
shendure: 10 sizes x 10 qualities x 4 algos x 1 signal (['author_day']) x 1 seeds ([42]) = 400 expected


## Generate all expected MI paths and check which exist

In [93]:
# Build all expected (dataset, size, quality, algo, signal, seed) -> MI file path
expected_rows = []
for ds, cfg in EXPECTED.items():
    for sz, q, algo, sig, sd in product(cfg['sizes'], cfg['qualities'], ALGOS, cfg['signals'], cfg['seeds']):
        suffix = '_geneformer' if algo == 'Geneformer' else ''
        stem = f'Y_{sig}_{q}{suffix}'
        mi_path = DATA_ROOT / ds / str(sz) / str(q) / 'results' / algo / 'model' / 'MI' / str(sd) / stem / 'lmi_mutual_information.txt'
        expected_rows.append({
            'dataset': ds, 'size': sz, 'quality': q, 'algorithm': algo,
            'signal': sig, 'seed': sd, 'path': str(mi_path),
        })

df_expected = pd.DataFrame(expected_rows)
print(f'Total expected: {len(df_expected)}')

# Check existence in parallel
paths = df_expected['path'].tolist()
print(f'Checking {len(paths)} paths...')
with ThreadPoolExecutor(max_workers=128) as pool:
    exists = list(tqdm(pool.map(os.path.exists, paths), total=len(paths), desc='Scanning'))

df_expected['exists'] = exists
n_found = df_expected['exists'].sum()
print(f'Found: {n_found}, Missing: {len(df_expected) - n_found}')

# Summary table: dataset x algorithm
pivot_mi = df_expected.groupby(['dataset', 'algorithm'])['exists'].agg(['sum', 'count'])
pivot_mi.columns = ['found', 'expected']
pivot_mi['missing'] = pivot_mi['expected'] - pivot_mi['found']
pivot_mi = pivot_mi.unstack('algorithm', fill_value=0)
display(pivot_mi)

Total expected: 4000
Checking 4000 paths...


Scanning: 100%|██████████| 4000/4000 [00:00<00:00, 426662.33it/s]

Found: 3969, Missing: 31


found                              expected       \
algorithm Geneformer  PCA RandomProjection SCVI Geneformer  PCA   
dataset                                                           
PBMC             300  300              300  300        300  300   
larry            300  300              300  300        300  300   
merfish          300  300              300  300        300  300   
shendure          89   92              100   88        100  100   

                                   missing                            
algorithm RandomProjection SCVI Geneformer PCA RandomProjection SCVI  
dataset                                                               
PBMC                   300  300          0   0                0    0  
larry                  300  300          0   0                0    0  
merfish                300  300          0   0                0    0  
shendure               100  100         11   8                0   12

## Read MI values and assemble `collect_mi_results_from_disk.csv`

In [94]:
df_found = df_expected[df_expected['exists']].copy()

def _read_mi(path):
    try:
        with open(path) as f:
            return float(f.read().strip())
    except Exception:
        return np.nan

print(f'Reading MI values from {len(df_found)} files...')
with ThreadPoolExecutor(max_workers=128) as pool:
    mi_values = list(tqdm(pool.map(_read_mi, df_found['path']), total=len(df_found), desc='Reading MI'))

df_found['mi_value'] = mi_values

# Save
out_cols = ['dataset', 'size', 'quality', 'algorithm', 'signal', 'seed', 'mi_value']
df_results = df_found[out_cols].sort_values(out_cols[:-1]).reset_index(drop=True)

OUT_PATH = Path('/home/igor/noise_scaling/modeling/Scaling-up-measurement-noise-scaling-laws/analysis/2026-04-08_veryfing_data_correctness/collect_mi_results_from_disk.csv')
df_results.to_csv(OUT_PATH, index=False)
print(f'Saved {len(df_results)} rows to {OUT_PATH}')
print(f'NaN mi_values: {df_results["mi_value"].isna().sum()}')
df_results.head(10)

Reading MI values from 3969 files...


Reading MI: 100%|██████████| 3969/3969 [00:00<00:00, 32210.94it/s]

Saved 3969 rows to /home/igor/noise_scaling/modeling/Scaling-up-measurement-noise-scaling-laws/analysis/2026-04-08_veryfing_data_correctness/collect_mi_results_from_disk.csv
NaN mi_values: 0


,dataset,size,quality,algorithm,signal,seed,mi_value
0,PBMC,100,0.001235,Geneformer,protein_counts,42,0.65307
1,PBMC,100,0.001235,Geneformer,protein_counts,2303,0.67208
2,PBMC,100,0.001235,Geneformer,protein_counts,2701,0.73480
3,PBMC,100,0.001235,PCA,protein_counts,42,0.35270
4,PBMC,100,0.001235,PCA,protein_counts,2303,0.37896
5,PBMC,100,0.001235,PCA,protein_counts,2701,0.34282
6,PBMC,100,0.001235,RandomProjection,protein_counts,42,0.25169
7,PBMC,100,0.001235,RandomProjection,protein_counts,2303,0.24459
8,PBMC,100,0.001235,RandomProjection,protein_counts,2701,0.25169
9,PBMC,100,0.001235,SCVI,protein_counts,42,0.25354


## Save missing configurations to `missing_mi_on_disk.csv`

In [95]:
df_missing = df_expected[~df_expected['exists']][['dataset', 'size', 'quality', 'algorithm', 'signal', 'seed']].copy()
df_missing = df_missing.sort_values(df_missing.columns.tolist()).reset_index(drop=True)

MISSING_PATH = Path('/home/igor/noise_scaling/modeling/Scaling-up-measurement-noise-scaling-laws/analysis/2026-04-08_veryfing_data_correctness/missing_mi_on_disk.csv')
df_missing.to_csv(MISSING_PATH, index=False)
print(f'Saved {len(df_missing)} missing configurations to {MISSING_PATH}')

if not df_missing.empty:
    pivot = df_missing.groupby(['dataset', 'algorithm']).size().unstack(fill_value=0)
    print('\nMissing MI counts per dataset x algorithm:')
    display(pivot)

Saved 31 missing configurations to /home/igor/noise_scaling/modeling/Scaling-up-measurement-noise-scaling-laws/analysis/2026-04-08_veryfing_data_correctness/missing_mi_on_disk.csv

Missing MI counts per dataset x algorithm:


algorithm,Geneformer,PCA,SCVI
dataset,,,
shendure,11,8,12


## Per-dataset, per-seed completeness breakdown

In [96]:
from IPython.display import display, Markdown

for ds, cfg in EXPECTED.items():
    seeds = cfg['seeds']
    n_combos = len(cfg['sizes']) * len(cfg['qualities']) * len(cfg['signals'])  # per algo per seed
    n_total = n_combos * len(ALGOS) * len(seeds)

    ds_exp = df_expected[df_expected['dataset'] == ds]
    n_have = ds_exp['exists'].sum()
    n_miss = n_total - n_have

    lines = []
    if n_miss == 0:
        lines.append(f'### {ds} — COMPLETE ({n_have}/{n_total})')
    else:
        lines.append(f'### {ds} — {n_miss} missing ({n_have}/{n_total})')

    lines.append(f'- {len(cfg["sizes"])} sizes × {len(cfg["qualities"])} qualities × {len(ALGOS)} algos '
                 f'× {len(cfg["signals"])} signal ({cfg["signals"][0]}) × {len(seeds)} seeds ({seeds})')

    # Table: rows = algorithm, columns = seeds
    lines.append(f'\n| Algorithm | ' + ' | '.join(f'seed {s}' for s in seeds) + ' | total |')
    lines.append('|---| ' + ' | '.join('---' for _ in seeds) + ' | --- |')

    for algo in ALGOS:
        cells = []
        row_total = 0
        for sd in seeds:
            n = ds_exp[(ds_exp['algorithm'] == algo) & (ds_exp['seed'] == sd) & ds_exp['exists']].shape[0]
            row_total += n
            cell = f'{n}/{n_combos}' if n == n_combos else f'**{n}/{n_combos}**'
            cells.append(cell)
        total_str = f'{row_total}/{n_combos * len(seeds)}' if row_total == n_combos * len(seeds) else f'**{row_total}/{n_combos * len(seeds)}**'
        cells.append(total_str)
        lines.append(f'| {algo} | ' + ' | '.join(cells) + ' |')

    # Totals row
    seed_totals = []
    for sd in seeds:
        st = ds_exp[(ds_exp['seed'] == sd) & ds_exp['exists']].shape[0]
        exp_per_seed = n_combos * len(ALGOS)
        seed_totals.append(f'{st}/{exp_per_seed}' if st == exp_per_seed else f'**{st}/{exp_per_seed}**')
    seed_totals.append(f'{n_have}/{n_total}' if n_have == n_total else f'**{n_have}/{n_total}**')
    lines.append(f'| **total** | ' + ' | '.join(seed_totals) + ' |')

    lines.append('')
    display(Markdown('\n'.join(lines)))

### PBMC — COMPLETE (1200/1200)
- 10 sizes × 10 qualities × 4 algos × 1 signal (protein_counts) × 3 seeds ([42, 2303, 2701])

| Algorithm | seed 42 | seed 2303 | seed 2701 | total |
|---| --- | --- | --- | --- |
| Geneformer | 100/100 | 100/100 | 100/100 | 300/300 |
| PCA | 100/100 | 100/100 | 100/100 | 300/300 |
| RandomProjection | 100/100 | 100/100 | 100/100 | 300/300 |
| SCVI | 100/100 | 100/100 | 100/100 | 300/300 |
| **total** | 400/400 | 400/400 | 400/400 | 1200/1200 |


### larry — COMPLETE (1200/1200)
- 10 sizes × 10 qualities × 4 algos × 1 signal (clone) × 3 seeds ([42, 1404, 2701])

| Algorithm | seed 42 | seed 1404 | seed 2701 | total |
|---| --- | --- | --- | --- |
| Geneformer | 100/100 | 100/100 | 100/100 | 300/300 |
| PCA | 100/100 | 100/100 | 100/100 | 300/300 |
| RandomProjection | 100/100 | 100/100 | 100/100 | 300/300 |
| SCVI | 100/100 | 100/100 | 100/100 | 300/300 |
| **total** | 400/400 | 400/400 | 400/400 | 1200/1200 |


### merfish — COMPLETE (1200/1200)
- 10 sizes × 10 qualities × 4 algos × 1 signal (ng_idx) × 3 seeds ([1404, 2303, 2701])

| Algorithm | seed 1404 | seed 2303 | seed 2701 | total |
|---| --- | --- | --- | --- |
| Geneformer | 100/100 | 100/100 | 100/100 | 300/300 |
| PCA | 100/100 | 100/100 | 100/100 | 300/300 |
| RandomProjection | 100/100 | 100/100 | 100/100 | 300/300 |
| SCVI | 100/100 | 100/100 | 100/100 | 300/300 |
| **total** | 400/400 | 400/400 | 400/400 | 1200/1200 |


### shendure — 31 missing (369/400)
- 10 sizes × 10 qualities × 4 algos × 1 signal (author_day) × 1 seeds ([42])

| Algorithm | seed 42 | total |
|---| --- | --- |
| Geneformer | **89/100** | **89/100** |
| PCA | **92/100** | **92/100** |
| RandomProjection | 100/100 | 100/100 |
| SCVI | **88/100** | **88/100** |
| **total** | **369/400** | **369/400** |


## Embeddings availability

In [97]:
emb_rows = []
for ds, cfg in EXPECTED.items():
    for sz, q, algo in product(cfg['sizes'], cfg['qualities'], ALGOS):
        emb_path = DATA_ROOT / ds / str(sz) / str(q) / 'results' / algo / 'model' / 'embeddings.csv'
        emb_rows.append({
            'dataset': ds, 'size': sz, 'quality': q, 'algorithm': algo,
            'path': str(emb_path),
        })

df_emb = pd.DataFrame(emb_rows)
print(f'Total expected embedding files: {len(df_emb)}')

with ThreadPoolExecutor(max_workers=128) as pool:
    emb_exists = list(tqdm(pool.map(os.path.exists, df_emb['path']), total=len(df_emb), desc='Scanning embeddings'))

df_emb['exists'] = emb_exists
n_emb_found = df_emb['exists'].sum()
print(f'Found: {n_emb_found}, Missing: {len(df_emb) - n_emb_found}')

# Summary table: dataset x algorithm
pivot_emb = df_emb.groupby(['dataset', 'algorithm'])['exists'].agg(['sum', 'count'])
pivot_emb.columns = ['found', 'expected']
pivot_emb['missing'] = pivot_emb['expected'] - pivot_emb['found']
pivot_emb = pivot_emb.unstack('algorithm', fill_value=0)
display(pivot_emb)

Total expected embedding files: 1600


Scanning embeddings: 100%|██████████| 1600/1600 [00:00<00:00, 433912.22it/s]

Found: 1571, Missing: 29


found                              expected       \
algorithm Geneformer  PCA RandomProjection SCVI Geneformer  PCA   
dataset                                                           
PBMC             100  100              100  100        100  100   
larry            100  100              100  100        100  100   
merfish          100  100              100  100        100  100   
shendure          89   92              100   90        100  100   

                                   missing                            
algorithm RandomProjection SCVI Geneformer PCA RandomProjection SCVI  
dataset                                                               
PBMC                   100  100          0   0                0    0  
larry                  100  100          0   0                0    0  
merfish                100  100          0   0                0    0  
shendure               100  100         11   8                0   10

## Models availability

In [98]:
MODEL_FILES = {
    'Geneformer': 'model.safetensors',
    'PCA': 'pca_model.pkl',
    'RandomProjection': 'random_projection.joblib',
    'SCVI': 'model.pt',
}

model_rows = []
for ds, cfg in EXPECTED.items():
    for sz, q, algo in product(cfg['sizes'], cfg['qualities'], ALGOS):
        model_path = DATA_ROOT / ds / str(sz) / str(q) / 'results' / algo / 'model' / MODEL_FILES[algo]
        model_rows.append({
            'dataset': ds, 'size': sz, 'quality': q, 'algorithm': algo,
            'path': str(model_path),
        })

df_model = pd.DataFrame(model_rows)
print(f'Total expected model files: {len(df_model)}')

with ThreadPoolExecutor(max_workers=128) as pool:
    model_exists = list(tqdm(pool.map(os.path.exists, df_model['path']), total=len(df_model), desc='Scanning models'))

df_model['exists'] = model_exists
n_model_found = df_model['exists'].sum()
print(f'Found: {n_model_found}, Missing: {len(df_model) - n_model_found}')

# Summary table: dataset x algorithm
pivot_model = df_model.groupby(['dataset', 'algorithm'])['exists'].agg(['sum', 'count'])
pivot_model.columns = ['found', 'expected']
pivot_model['missing'] = pivot_model['expected'] - pivot_model['found']
pivot_model = pivot_model.unstack('algorithm', fill_value=0)
display(pivot_model)

Total expected model files: 1600


Scanning models: 100%|██████████| 1600/1600 [00:00<00:00, 439136.66it/s]

Found: 1572, Missing: 28


found                              expected       \
algorithm Geneformer  PCA RandomProjection SCVI Geneformer  PCA   
dataset                                                           
PBMC             100  100              100  100        100  100   
larry            100  100              100  100        100  100   
merfish          100  100              100  100        100  100   
shendure          90   92              100   90        100  100   

                                   missing                            
algorithm RandomProjection SCVI Geneformer PCA RandomProjection SCVI  
dataset                                                               
PBMC                   100  100          0   0                0    0  
larry                  100  100          0   0                0    0  
merfish                100  100          0   0                0    0  
shendure               100  100         10   8                0   10